# 13. Log-Mel CNN — In-domain Baseline

이 노트북에서는 10초 오디오 segment를 **Log-Mel Spectrogram**으로 변환한 뒤,
경량 CNN이 REAL / FAKE를 직접 학습하도록 한다.

기존 Logistic Regression / RBF-SVM과의 차이는 다음과 같다.

```text
LR / SVM
오디오
→ 사람이 정한 266개 특징
→ 모델

CNN
오디오
→ Log-Mel Spectrogram
→ CNN이 시간-주파수 패턴을 직접 학습
```

## 입력 설정

- Sample rate: 24,000 Hz
- Segment: 10초
- Mel bins: 128
- `n_fft = 1024`
- `hop_length = 240`
- `fmax = 12,000 Hz`

## 학습 원칙

- 기존 `original_audio` 기준 Train / Val / Test split 그대로 사용
- Train / Val / Test를 다시 섞지 않음
- 클래스 불균형을 weighted BCE로 보정
- AdamW optimizer
- Validation **Track-level EER** 기준 Early Stopping
- 최종 threshold는 Validation에서 결정
- Test threshold 재튜닝 금지
- Segment-level + Track-level 모두 평가


In [1]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"

LOGMEL_DIR = PROJECT_ROOT / "data/processed/logmel"
LOGMEL_DIR.mkdir(parents=True, exist_ok=True)

LOGMEL_PATH = LOGMEL_DIR / "logmel_10s_float16.npy"
DONE_PATH = LOGMEL_DIR / "logmel_10s_done.npy"
INDEX_PATH = LOGMEL_DIR / "logmel_10s_index.csv"

RESULT_DIR = PROJECT_ROOT / "results/cnn"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints/cnn"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SR = 24_000
SEGMENT_SEC = 10.0
TARGET_SAMPLES = int(SR * SEGMENT_SEC)

N_FFT = 1024
HOP_LENGTH = 240
N_MELS = 128
FMAX = 12_000

# librosa center=True 기준
N_FRAMES = 1 + TARGET_SAMPLES // HOP_LENGTH

RANDOM_STATE = 42

print("SEGMENT_PATH :", SEGMENT_PATH)
print("LOGMEL_PATH  :", LOGMEL_PATH)
print("Shape target :", (N_MELS, N_FRAMES))


SEGMENT_PATH : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/segment_manifest_10s.csv
LOGMEL_PATH  : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/logmel/logmel_10s_float16.npy
Shape target : (128, 1001)


## 1. PyTorch 확인 및 Device 선택

Mac에서는 Apple Silicon/MPS가 사용 가능하면 자동으로 MPS를 사용한다.

우선순위:

```text
CUDA → MPS → CPU
```


In [2]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except ImportError:
    raise ImportError(
        "PyTorch가 설치되어 있지 않습니다. "
        "새 코드 셀에서 `%pip install torch`를 실행한 뒤 커널을 재시작하세요."
    )

print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)


: 

## 2. 재현성 설정


In [ ]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print("Random seed:", RANDOM_STATE)


## 3. Segment Manifest 로드

07단계에서 확정한 10,077개 segment를 사용한다.


In [ ]:
segments = pd.read_csv(SEGMENT_PATH).reset_index(drop=True)

print("===== SEGMENT MANIFEST =====")
print("Rows             :", len(segments))
print("Unique segment_id:", segments["segment_id"].nunique())
print("Tracks           :", segments["track_sample_id"].nunique())
print("Original groups  :", segments["original_audio"].nunique())

print("\nSplit:")
print(segments["split"].value_counts())

print("\nLabel:")
print(segments["label"].value_counts())

assert len(segments) == 10077
assert segments["segment_id"].nunique() == 10077


# Part A. Log-Mel Spectrogram Cache

10,077개 segment에 대해 매 epoch마다 spectrogram을 다시 계산하면 매우 느리다.

따라서 처음 한 번만 Log-Mel을 계산하여 `float16` NumPy 파일로 저장하고,
CNN 학습에서는 이를 바로 읽는다.

예상 저장 용량은 약 **2.6 GB**이다.

중간에 중단되어도 `done` mask를 이용해 이어서 계산할 수 있다.


## 4. 10초 waveform 로딩 함수


In [ ]:
def load_segment_waveform(row):
    full_path = PROJECT_ROOT / row["audio_path"]

    if not full_path.exists():
        raise FileNotFoundError(full_path)

    y, _ = librosa.load(
        full_path,
        sr=SR,
        mono=True,
        offset=float(row["start_sec"]),
        duration=SEGMENT_SEC,
    )

    y = np.asarray(y, dtype=np.float32)

    if len(y) < TARGET_SAMPLES:
        y = np.pad(
            y,
            (0, TARGET_SAMPLES - len(y)),
            mode="constant",
        )
    elif len(y) > TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]

    if len(y) != TARGET_SAMPLES:
        raise RuntimeError(
            f"waveform length mismatch: {len(y)}"
        )

    return y


## 5. Log-Mel 변환 함수

각 segment에서:

```text
waveform
→ Mel Spectrogram
→ dB scale
→ [-80, 0] dB
→ [-1, 1] 범위로 정규화
```

이 정규화는 각 segment에 동일한 고정 수식을 적용하므로
Train / Val / Test 사이의 통계적 leakage가 발생하지 않는다.


In [ ]:
def waveform_to_logmel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmax=FMAX,
        power=2.0,
        center=True,
    )

    logmel = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=80.0,
    )

    # [-80, 0] → [-1, 1]
    logmel = (logmel + 40.0) / 40.0

    if logmel.shape != (N_MELS, N_FRAMES):
        raise RuntimeError(
            f"Unexpected logmel shape: {logmel.shape}"
        )

    if not np.isfinite(logmel).all():
        raise ValueError("Log-Mel contains NaN or Inf")

    return logmel.astype(np.float32)


## 6. 첫 Segment 시험

전체 캐시를 만들기 전에 한 개만 확인한다.


In [ ]:
test_row = segments.iloc[0]

test_y = load_segment_waveform(test_row)
test_logmel = waveform_to_logmel(test_y)

print("Waveform shape:", test_y.shape)
print("Log-Mel shape :", test_logmel.shape)
print("dtype         :", test_logmel.dtype)
print("min / max     :", test_logmel.min(), test_logmel.max())


In [ ]:
plt.figure(figsize=(12, 4))
plt.imshow(
    test_logmel,
    origin="lower",
    aspect="auto",
)
plt.title("Example 10-second Log-Mel Spectrogram")
plt.xlabel("Time frame")
plt.ylabel("Mel bin")
plt.colorbar()
plt.tight_layout()
plt.show()


## 7. Log-Mel Cache 초기화 / 재개

`logmel_10s_float16.npy`에는 spectrogram을 저장하고,
`logmel_10s_done.npy`에는 각 행의 계산 완료 여부를 저장한다.


In [ ]:
EXPECTED_SHAPE = (
    len(segments),
    N_MELS,
    N_FRAMES,
)

if LOGMEL_PATH.exists():
    logmel_memmap = np.lib.format.open_memmap(
        LOGMEL_PATH,
        mode="r+",
    )

    if logmel_memmap.shape != EXPECTED_SHAPE:
        raise RuntimeError(
            f"Existing cache shape mismatch: "
            f"{logmel_memmap.shape} != {EXPECTED_SHAPE}"
        )
else:
    logmel_memmap = np.lib.format.open_memmap(
        LOGMEL_PATH,
        mode="w+",
        dtype=np.float16,
        shape=EXPECTED_SHAPE,
    )

if DONE_PATH.exists():
    done_mask = np.load(DONE_PATH)

    if len(done_mask) != len(segments):
        raise RuntimeError("done mask length mismatch")
else:
    done_mask = np.zeros(
        len(segments),
        dtype=bool,
    )
    np.save(DONE_PATH, done_mask)

print("Cache shape  :", logmel_memmap.shape)
print("Completed    :", int(done_mask.sum()))
print("Remaining    :", int((~done_mask).sum()))


## 8. 전체 Log-Mel Cache 생성

첫 실행에서 시간이 가장 오래 걸리는 부분이다.

100개마다 진행 상황과 done mask를 저장한다.
중단되면 같은 셀을 다시 실행하면 남은 segment부터 계속한다.


In [ ]:
CACHE_SAVE_EVERY = 100

remaining_indices = np.where(~done_mask)[0]

start_time = time.time()

for n, idx in enumerate(remaining_indices, start=1):
    row = segments.iloc[idx]

    y = load_segment_waveform(row)
    logmel = waveform_to_logmel(y)

    logmel_memmap[idx] = logmel.astype(np.float16)
    done_mask[idx] = True

    if n % CACHE_SAVE_EVERY == 0 or n == len(remaining_indices):
        logmel_memmap.flush()
        np.save(DONE_PATH, done_mask)

        elapsed = time.time() - start_time

        print(
            f"this run: {n}/{len(remaining_indices)} | "
            f"total done: {int(done_mask.sum())}/{len(done_mask)} | "
            f"elapsed: {elapsed/60:.1f} min"
        )

np.save(DONE_PATH, done_mask)

segments[
    [
        "segment_id",
        "track_sample_id",
        "original_audio",
        "label",
        "label_id",
        "genre",
        "generator",
        "split",
    ]
].to_csv(
    INDEX_PATH,
    index=True,
    index_label="logmel_index",
    encoding="utf-8-sig",
)

print("\nLog-Mel cache complete:", bool(done_mask.all()))


## 9. Cache QC


In [ ]:
done_mask = np.load(DONE_PATH)

print("Rows           :", len(done_mask))
print("Completed      :", int(done_mask.sum()))
print("Incomplete     :", int((~done_mask).sum()))
print("Cache shape    :", logmel_memmap.shape)
print("Cache dtype    :", logmel_memmap.dtype)

cache_qc_pass = (
    len(done_mask) == 10077
    and bool(done_mask.all())
    and logmel_memmap.shape
        == (10077, 128, N_FRAMES)
)

print("Log-Mel Cache QC PASS:", cache_qc_pass)


# Part B. CNN Dataset / DataLoader


## 10. PyTorch Dataset

각 sample은:

```text
X: [1, 128, 1001]
y: 0(REAL) / 1(FAKE)
```

형태다.


In [ ]:
class LogMelDataset(Dataset):
    def __init__(self, metadata, memmap_path):
        self.metadata = metadata.reset_index(drop=False).copy()
        self.memmap_path = memmap_path

        # 각 Dataset process에서 mmap을 직접 읽음
        self.data = np.load(
            self.memmap_path,
            mmap_mode="r",
        )

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        original_index = int(row["index"])

        x = np.asarray(
            self.data[original_index],
            dtype=np.float32,
        )

        x = torch.from_numpy(
            x.copy()
        ).unsqueeze(0)

        y = torch.tensor(
            float(row["label_id"]),
            dtype=torch.float32,
        )

        return x, y, original_index


train_meta = segments[
    segments["split"] == "train"
].copy()

val_meta = segments[
    segments["split"] == "val"
].copy()

test_meta = segments[
    segments["split"] == "test"
].copy()

train_ds = LogMelDataset(
    train_meta,
    LOGMEL_PATH,
)

val_ds = LogMelDataset(
    val_meta,
    LOGMEL_PATH,
)

test_ds = LogMelDataset(
    test_meta,
    LOGMEL_PATH,
)

print("Train:", len(train_ds))
print("Val  :", len(val_ds))
print("Test :", len(test_ds))


## 11. DataLoader

Mac/Jupyter 안정성을 위해 기본 `num_workers=0`을 사용한다.
MPS 메모리가 부족하면 `BATCH_SIZE`를 16으로 낮춘다.


In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

xb, yb, idxb = next(iter(train_loader))

print("Batch X:", xb.shape)
print("Batch y:", yb.shape)


# Part C. Lightweight Log-Mel CNN


## 12. CNN 구조

4개의 convolution block을 사용한다.

```text
1 channel
→ 16
→ 32
→ 64
→ 128
→ Adaptive Average Pooling
→ Dropout
→ Binary Logit
```

파라미터 수는 2M보다 훨씬 작도록 제한한다.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class LogMelCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(1, 16),
            ConvBlock(16, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
        )

        self.pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)

        return x.squeeze(1)


model = LogMelCNN().to(DEVICE)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(model)
print("\nTotal params    :", total_params)
print("Trainable params:", trainable_params)

assert total_params < 2_000_000


## 13. 클래스 불균형 가중치

FAKE가 약 91%, REAL이 약 9%이므로 그대로 BCE를 사용하면
다수 클래스인 FAKE에 지나치게 유리할 수 있다.

`FAKE = positive class`이므로:

```text
pos_weight
= Train REAL 수 / Train FAKE 수
```

를 사용하여 두 클래스의 총 loss 기여도를 비슷하게 맞춘다.


In [ ]:
train_real = int(
    (train_meta["label"] == "REAL").sum()
)

train_fake = int(
    (train_meta["label"] == "FAKE").sum()
)

POS_WEIGHT = (
    train_real / train_fake
)

print("Train REAL:", train_real)
print("Train FAKE:", train_fake)
print("pos_weight:", POS_WEIGHT)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [POS_WEIGHT],
        dtype=torch.float32,
        device=DEVICE,
    )
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)


# Part D. Evaluation Functions


In [ ]:
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)


def find_eer_threshold(y_true, scores):
    fpr, tpr, thresholds = roc_curve(
        y_true,
        scores,
        pos_label=1,
    )

    fnr = 1.0 - tpr
    valid = np.isfinite(thresholds)

    fpr = fpr[valid]
    fnr = fnr[valid]
    thresholds = thresholds[valid]

    idx = np.argmin(
        np.abs(fpr - fnr)
    )

    return {
        "eer": float(
            (fpr[idx] + fnr[idx]) / 2.0
        ),
        "threshold": float(
            thresholds[idx]
        ),
        "fpr_at_eer": float(fpr[idx]),
        "fnr_at_eer": float(fnr[idx]),
    }


def evaluate_scores(
    y_true,
    scores,
    threshold,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )
    scores = np.asarray(
        scores,
        dtype=float,
    )

    y_pred = (
        scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    eer_info = find_eer_threshold(
        y_true,
        scores
    )

    return {
        "roc_auc": float(
            roc_auc_score(
                y_true,
                scores
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                scores
            )
        ),
        "eer": float(
            eer_info["eer"]
        ),
        "threshold_used": float(
            threshold
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "real_fpr": float(
            fp / (fp + tn)
        ),
        "fake_miss_rate": float(
            fn / (fn + tp)
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def make_track_scores(
    metadata,
    scores,
):
    temp = metadata[
        [
            "track_sample_id",
            "original_audio",
            "label",
            "label_id",
            "genre",
            "generator",
            "split",
        ]
    ].copy()

    temp["score"] = scores

    return (
        temp
        .groupby(
            "track_sample_id",
            as_index=False
        )
        .agg(
            original_audio=(
                "original_audio",
                "first"
            ),
            label=("label", "first"),
            label_id=("label_id", "first"),
            genre=("genre", "first"),
            generator=("generator", "first"),
            split=("split", "first"),
            segment_count=("score", "size"),
            score=("score", "mean"),
        )
    )


## 14. Prediction 함수


In [ ]:
@torch.no_grad()
def predict_loader(
    model,
    loader,
):
    model.eval()

    all_scores = []
    all_labels = []
    all_indices = []

    for x, y, original_idx in loader:
        x = x.to(
            DEVICE,
            non_blocking=False
        )

        logits = model(x)

        scores = torch.sigmoid(
            logits
        ).cpu().numpy()

        all_scores.append(scores)
        all_labels.append(
            y.numpy()
        )
        all_indices.append(
            original_idx.numpy()
        )

    return (
        np.concatenate(all_labels),
        np.concatenate(all_scores),
        np.concatenate(all_indices),
    )


# Part E. CNN Training


## 15. 학습 함수

Early stopping 기준은 **Validation Track-level EER**이다.

같은 곡의 여러 segment score를 평균한 뒤 계산한다.


In [ ]:
MAX_EPOCHS = 20
PATIENCE = 4

BEST_MODEL_PATH = (
    CHECKPOINT_DIR
    / "logmel_cnn_best.pt"
)

history_rows = []

best_val_track_eer = np.inf
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    model.train()

    running_loss = 0.0
    seen = 0

    for x, y, _ in train_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(
            logits,
            y
        )

        loss.backward()
        optimizer.step()

        batch_size = x.size(0)

        running_loss += (
            loss.item()
            * batch_size
        )
        seen += batch_size

    train_loss = (
        running_loss / seen
    )

    # Validation
    val_y, val_scores, val_indices = (
        predict_loader(
            model,
            val_loader
        )
    )

    val_meta_ordered = (
        segments.iloc[val_indices]
        .reset_index(drop=True)
    )

    val_segment_eer = (
        find_eer_threshold(
            val_y,
            val_scores
        )
    )

    val_track_df = make_track_scores(
        val_meta_ordered,
        val_scores,
    )

    val_track_eer = (
        find_eer_threshold(
            val_track_df["label_id"],
            val_track_df["score"],
        )
    )

    val_roc_auc = roc_auc_score(
        val_y,
        val_scores
    )

    elapsed = time.time() - epoch_start

    history_rows.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_segment_roc_auc": val_roc_auc,
        "val_segment_eer": val_segment_eer["eer"],
        "val_track_eer": val_track_eer["eer"],
        "elapsed_sec": elapsed,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"loss={train_loss:.4f} | "
        f"val seg AUC={val_roc_auc:.4f} | "
        f"val seg EER={val_segment_eer['eer']:.4f} | "
        f"val track EER={val_track_eer['eer']:.4f} | "
        f"{elapsed:.1f}s"
    )

    # Early stopping
    if (
        val_track_eer["eer"]
        < best_val_track_eer - 1e-5
    ):
        best_val_track_eer = (
            val_track_eer["eer"]
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),
                "epoch": epoch,
                "best_val_track_eer":
                    best_val_track_eer,
                "config": {
                    "sr": SR,
                    "segment_sec":
                        SEGMENT_SEC,
                    "n_fft": N_FFT,
                    "hop_length":
                        HOP_LENGTH,
                    "n_mels": N_MELS,
                    "fmax": FMAX,
                },
            },
            BEST_MODEL_PATH,
        )

        print(
            "  -> saved best model"
        )

    else:
        epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= PATIENCE
        ):
            print(
                f"Early stopping at epoch "
                f"{epoch}"
            )
            break

history = pd.DataFrame(
    history_rows
)

history.to_csv(
    RESULT_DIR / "training_history.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "\nBest Validation Track EER:",
    best_val_track_eer
)


## 16. 학습 곡선 확인


In [ ]:
display(history)

ax = history.plot(
    x="epoch",
    y=[
        "val_segment_eer",
        "val_track_eer",
    ],
    marker="o",
    figsize=(8, 5),
    title="CNN Validation EER",
)

ax.set_ylabel("EER")
plt.tight_layout()
plt.show()


# Part F. Best Model Final Evaluation


## 17. Best checkpoint 로드


In [ ]:
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print(
    "Best epoch:",
    checkpoint["epoch"]
)

print(
    "Best val track EER:",
    checkpoint[
        "best_val_track_eer"
    ]
)


## 18. Validation score와 threshold 결정

최종 best model에서:

- Segment-level threshold
- Track-level threshold

를 각각 Validation EER 기준으로 결정한다.


In [ ]:
val_y, val_scores, val_indices = (
    predict_loader(
        model,
        val_loader
    )
)

val_meta_ordered = (
    segments.iloc[val_indices]
    .reset_index(drop=True)
)

val_segment_eer = find_eer_threshold(
    val_y,
    val_scores
)

val_track = make_track_scores(
    val_meta_ordered,
    val_scores,
)

val_track_eer = find_eer_threshold(
    val_track["label_id"],
    val_track["score"],
)

SEGMENT_THRESHOLD = (
    val_segment_eer["threshold"]
)

TRACK_THRESHOLD = (
    val_track_eer["threshold"]
)

print("===== VAL SEGMENT EER =====")
print(val_segment_eer)

print("\n===== VAL TRACK EER =====")
print(val_track_eer)


## 19. Test 평가

Test threshold는 다시 튜닝하지 않고
Validation에서 정한 값을 그대로 사용한다.


In [ ]:
test_y, test_scores, test_indices = (
    predict_loader(
        model,
        test_loader
    )
)

test_meta_ordered = (
    segments.iloc[test_indices]
    .reset_index(drop=True)
)

test_segment_metrics = (
    evaluate_scores(
        test_y,
        test_scores,
        SEGMENT_THRESHOLD,
    )
)

test_track = make_track_scores(
    test_meta_ordered,
    test_scores,
)

test_track_metrics = (
    evaluate_scores(
        test_track["label_id"],
        test_track["score"],
        TRACK_THRESHOLD,
    )
)

val_segment_metrics = (
    evaluate_scores(
        val_y,
        val_scores,
        SEGMENT_THRESHOLD,
    )
)

val_track_metrics = (
    evaluate_scores(
        val_track["label_id"],
        val_track["score"],
        TRACK_THRESHOLD,
    )
)

cnn_results = pd.DataFrame([
    {
        "model": "LogMelCNN",
        "level": "segment",
        "split": "val",
        **val_segment_metrics,
    },
    {
        "model": "LogMelCNN",
        "level": "segment",
        "split": "test",
        **test_segment_metrics,
    },
    {
        "model": "LogMelCNN",
        "level": "track",
        "split": "val",
        **val_track_metrics,
    },
    {
        "model": "LogMelCNN",
        "level": "track",
        "split": "test",
        **test_track_metrics,
    },
])

display(
    cnn_results[
        [
            "model",
            "level",
            "split",
            "roc_auc",
            "pr_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
            "threshold_used",
        ]
    ].round(4)
)


## 20. CNN Test confusion 정보


In [ ]:
print("===== SEGMENT TEST =====")
print(test_segment_metrics)

print("\n===== TRACK TEST =====")
print(test_track_metrics)


# Part G. RBF-SVM과 비교


## 21. 기존 baseline 결과와 비교

현재 가장 강했던 handcrafted baseline인
RBF-SVM과 CNN을 직접 비교한다.


In [ ]:
BASELINE_METRICS_PATH = (
    PROJECT_ROOT
    / "results/baseline/baseline_metrics.csv"
)

if BASELINE_METRICS_PATH.exists():
    baseline_metrics = pd.read_csv(
        BASELINE_METRICS_PATH
    )

    svm_test = baseline_metrics[
        (baseline_metrics["model"] == "RBF-SVM")
        & (baseline_metrics["split"] == "test")
    ].copy()

    cnn_test = cnn_results[
        cnn_results["split"] == "test"
    ].copy()

    comparison = pd.concat(
        [
            svm_test,
            cnn_test,
        ],
        ignore_index=True,
    )

    display(
        comparison[
            [
                "model",
                "level",
                "roc_auc",
                "eer",
                "balanced_accuracy",
                "macro_f1",
                "real_fpr",
                "fake_miss_rate",
            ]
        ].round(4)
    )
else:
    print(
        "Baseline metrics not found:",
        BASELINE_METRICS_PATH
    )


## 22. Test prediction 저장

이후 generator별 / genre별 / unseen / MP3 robustness 비교에 사용한다.


In [ ]:
segment_predictions = (
    test_meta_ordered[
        [
            "segment_id",
            "track_sample_id",
            "original_audio",
            "label",
            "label_id",
            "genre",
            "generator",
            "split",
        ]
    ].copy()
)

segment_predictions[
    "cnn_score"
] = test_scores

segment_predictions[
    "cnn_pred"
] = (
    test_scores
    >= SEGMENT_THRESHOLD
).astype(int)

segment_predictions.to_csv(
    RESULT_DIR
    / "test_segment_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)


track_predictions = (
    test_track.copy()
)

track_predictions[
    "cnn_pred"
] = (
    track_predictions["score"]
    >= TRACK_THRESHOLD
).astype(int)

track_predictions = (
    track_predictions.rename(
        columns={
            "score": "cnn_score"
        }
    )
)

track_predictions.to_csv(
    RESULT_DIR
    / "test_track_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved CNN predictions.")


## 23. Metrics / Threshold 저장


In [ ]:
cnn_results.to_csv(
    RESULT_DIR / "cnn_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

threshold_info = {
    "segment_eer_threshold":
        float(SEGMENT_THRESHOLD),
    "track_eer_threshold":
        float(TRACK_THRESHOLD),
    "segment_val_eer":
        float(val_segment_eer["eer"]),
    "track_val_eer":
        float(val_track_eer["eer"]),
    "best_epoch":
        int(checkpoint["epoch"]),
}

with open(
    RESULT_DIR / "cnn_thresholds.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        threshold_info,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved metrics and thresholds.")


## 24. 최종 CNN QC


In [ ]:
qc_summary = pd.DataFrame({
    "check": [
        "segment_rows",
        "logmel_cache_complete",
        "train_segments",
        "val_segments",
        "test_segments",
        "model_params",
        "best_model_exists",
        "segment_threshold_finite",
        "track_threshold_finite",
        "test_track_rows",
        "result_rows",
    ],
    "value": [
        len(segments),
        bool(done_mask.all()),
        len(train_meta),
        len(val_meta),
        len(test_meta),
        total_params,
        BEST_MODEL_PATH.exists(),
        np.isfinite(SEGMENT_THRESHOLD),
        np.isfinite(TRACK_THRESHOLD),
        len(test_track),
        len(cnn_results),
    ],
})

display(qc_summary)

cnn_qc_pass = (
    len(segments) == 10077
    and bool(done_mask.all())
    and len(train_meta) == 6967
    and len(val_meta) == 1538
    and len(test_meta) == 1572
    and total_params < 2_000_000
    and BEST_MODEL_PATH.exists()
    and np.isfinite(SEGMENT_THRESHOLD)
    and np.isfinite(TRACK_THRESHOLD)
    and len(test_track) == 539
    and len(cnn_results) == 4
)

print("===== FINAL RESULT =====")
print("CNN Core QC PASS:", cnn_qc_pass)


## 다음 단계

CNN in-domain baseline이 완료되면 다음 순서로 확장한다.

1. CNN generator별 / genre별 subgroup 분석
2. CNN unseen-generator 실험
   - MusicGen holdout
   - Udio holdout
3. CNN MP3 robustness
   - Original
   - MP3 128 kbps
   - MP3 64 kbps
4. RBF-SVM vs CNN 종합 비교

이렇게 하면 최종적으로:

```text
Handcrafted RBF-SVM
vs
Log-Mel CNN
```

을 세 환경에서 비교할 수 있다.

- In-domain
- Unseen generator
- MP3 compression


## 최신 실행 결과 요약 (2026-09-13)

Log-Mel CNN 최종 Test 결과다.

| 수준 | ROC-AUC | EER | Balanced Accuracy | Macro-F1 |
|---|---:|---:|---:|---:|
| Segment | 0.9520 | 0.1349 | 0.8791 | 0.7795 |
| Track | **0.9772** | **0.1042** | **0.9111** | **0.8535** |

- Best epoch는 18이며, Validation EER threshold는 Segment 0.04316, Track 0.09354다.
- Track CNN은 RBF-SVM Track의 ROC-AUC 0.9644보다 높은 in-domain 성능을 보였다.
- 모델, threshold, history, Test 예측을 `checkpoints/cnn/`과 `results/cnn/`에 저장했다.

**최종 상태: CNN Core QC 통과.**
